In [ ]:
import signal, time as _t
_T0 = _t.time()
def _cap(*a): raise TimeoutError("40 min cap")
signal.signal(signal.SIGALRM, _cap); signal.alarm(40 * 60)
import base64, io, zipfile, subprocess, sys, os, time, json
os.environ["HF_HUB_DISABLE_XET"] = "1"
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers>=5.17", "gguf>=0.10", "accelerate", "git+https://github.com/huggingface/diffusers"], check=False)
os.makedirs("/kaggle/working/code", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall("/kaggle/working/code")
sys.path.insert(0, "/kaggle/working/code/prototype")
from PIL import Image
from layerproto.serve import qwen21
import torch
pipe = qwen21.load()
DATA = {"original": "...", "naive": "...", "naive2": "...", "scene_nofox": "...", "fox1": "...", "fox2": "..."}
img = lambda k: Image.open(io.BytesIO(base64.b64decode(DATA[k])))
RGBA = "This is an RGBA image with transparency. {}. The image has alpha channel and the background is transparent."
os.makedirs("/kaggle/working/harm", exist_ok=True)
for k in DATA:
    img(k).convert("RGBA" if k.startswith("fox") else "RGB").save(f"/kaggle/working/harm/input_{k}.png")
results = []
for name, ins, prompt, seed, rgba in [["control", ["naive"], "Keep the picture exactly the same. Do not change anything.", 42, false], ["harmonise_s42", ["naive"], "The red fox in this picture was pasted in from another picture. Make it look naturally part of the scene: match the lighting and colours of the surroundings, add a soft cast shadow on the path under the fox, and keep every other part of the picture exactly the same, including the fox's shape, pose and markings.", 42, false], ["harmonise_s7", ["naive"], "The red fox in this picture was pasted in from another picture. Make it look naturally part of the scene: match the lighting and colours of the surroundings, add a soft cast shadow on the path under the fox, and keep every other part of the picture exactly the same, including the fox's shape, pose and markings.", 7, false], ["shadow_only", ["naive"], "Add only a soft cast shadow of the red fox on the cobblestone path below it. Keep everything else exactly the same.", 42, false], ["refs", ["fox1", "scene_nofox"], "Place the red fox from the first image into the second image exactly where it is in the first image: same position, same size, same pose. Light it like the second image and add a soft shadow on the path. Keep the rest of the second image exactly the same.", 42, false], ["layer_relight", ["fox1"], "Relight this fox as if it stands on a sunlit cobblestone path in a sunny meadow, soft light from the upper left, keep its exact shape, pose and markings", 42, true], ["harder_control", ["naive2"], "Keep the picture exactly the same. Do not change anything.", 42, false], ["harder_harmonise", ["naive2"], "The red fox in this picture was pasted in from another picture. Make it look naturally part of the scene: match the lighting and colours of the surroundings, add a soft cast shadow on the path under the fox, and keep every other part of the picture exactly the same, including the fox's shape, pose and markings.", 42, false]]:
    row = dict(job=name, inputs=ins, seed=seed)
    try:
        images = [img(k).convert("RGBA" if k.startswith("fox") else "RGB") for k in ins]
        t0 = time.time()
        with torch.inference_mode():
            out = pipe(prompt=RGBA.format(prompt) if rgba else prompt, image=images, width=512, height=512, output_resolution=512,
                       use_kv_cache=True, generator=torch.Generator(pipe._execution_device).manual_seed(seed)).images[0]
        row["compute_s"] = round(time.time() - t0, 1)
        out.save(f"/kaggle/working/harm/{name}.png")
        row["mode"] = out.mode
    except Exception as e:
        import traceback; traceback.print_exc(); row["error"] = repr(e)[:300]
    print(json.dumps(row), flush=True); results.append(row)
    json.dump(results, open("/kaggle/working/harm/results.json", "w"), indent=1)
print("TRIAL DONE", round(time.time() - _T0), "s", flush=True)